In [ ]:
import aspose.slides as slides
from pathlib import Path

ppt_path = r"C:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\runs\run_20260608_145816\outputs\20260615_HOSPITAL_TEMPLATE_CurrentState_insert.pptx"
output_txt = Path(ppt_path).with_suffix(".object_inventory.txt")


def extract_text(shape):
    try:
        if hasattr(shape, "text_frame") and shape.text_frame:
            texts = []
            for para in shape.text_frame.paragraphs:
                texts.append(para.text.strip())
            return "\n".join(texts)
    except Exception:
        pass
    return None


def write_shape_block(f, scope, slide_idx, shape_idx, shape):
    text = extract_text(shape)

    f.write("=" * 80 + "\n")
    f.write(f"Scope        : {scope}\n")
    f.write(f"Slide Index  : {slide_idx}\n")
    f.write(f"Shape Index  : {shape_idx}\n")
    f.write(f"Shape Type   : {type(shape)}\n")
    f.write(f"Shape Name   : {getattr(shape, 'name', None)}\n")
    f.write(f"Has Text     : {text is not None}\n")

    if text:
        f.write("\n--- TEXT CONTENT ---\n")
        f.write(text + "\n")

    f.write("\n")


with slides.Presentation(ppt_path) as presentation, open(output_txt, "w", encoding="utf-8") as f:

    f.write("\n" + "#" * 100 + "\n")
    f.write("POWERPOINT OBJECT INVENTORY\n")
    f.write("#" * 100 + "\n\n")

    # ✅ Slides
    for i, slide in enumerate(presentation.slides):

        f.write(f"\n\n{'#'*40}\nSLIDE {i}\n{'#'*40}\n")

        for j, shape in enumerate(slide.shapes):
            write_shape_block(f, "slide", i, j, shape)

        # ✅ Notes slides (correct API)
        try:
            if slide.notes_slide_manager is not None:
                notes = slide.notes_slide_manager.notes_slide
                if notes is not None:
                    f.write(f"\n--- NOTES SLIDE ({i}) ---\n")
                    for j, shape in enumerate(notes.shapes):
                        write_shape_block(f, "notes", i, j, shape)
        except Exception:
            pass

    # ✅ Layouts
    f.write(f"\n\n{'#'*40}\nLAYOUT SLIDES\n{'#'*40}\n")
    for i, layout in enumerate(presentation.layout_slides):
        f.write(f"\n--- LAYOUT {i} ---\n")
        for j, shape in enumerate(layout.shapes):
            write_shape_block(f, "layout", i, j, shape)

    # ✅ Masters
    f.write(f"\n\n{'#'*40}\nMASTER SLIDES\n{'#'*40}\n")
    for i, master in enumerate(presentation.masters):
        f.write(f"\n--- MASTER {i} ---\n")
        for j, shape in enumerate(master.shapes):
            write_shape_block(f, "master", i, j, shape)


print(f"✅ Text inspection file created:\n{output_txt}")